# Codificação de Sinais Multimedia

## Trabalho Laboratorial 3

- 52146 - Margarida Andrade
- 52345 - Jorge Gonçalves
- 52708 - Helena Nina

In [1]:
import numpy as np
import matplotlib.pyplot as plt

## Exercício 1


### Exercício 1

Nesta alínea é construída uma tabela de Huffman a partir de símbolos e respetivos pesos (probabilidades ou ocorrências). A função devolve uma tabela com o código binário associado a cada símbolo e pode ser aplicada diretamente a um histograma de frequência.

In [2]:
from collections import Counter
from heapq import heappop, heappush
from itertools import count
from pathlib import Path


def gen_huff_table(symbols, probabilities=None):
    """Gera um dicionario simbolo -> codigo de Huffman.
        symbols: lista de simbolos ou dicionario {simbolo: probabilidade}
        probabilities: lista de probabilidades (opcional se symbols é dict)
    """
    if isinstance(symbols, dict):
        freq = symbols
    else:
        if probabilities is None:
            raise ValueError("probabilities deve ser fornecido quando symbols nao e um dicionario")
        freq = dict(zip(symbols, probabilities))
    
    heap = []
    order = count()
    for symbol, weight in freq.items():
        heappush(heap, (float(weight), next(order), symbol))
    
    if len(heap) == 1:
        return {heap[0][2]: "0"}
    
    while len(heap) > 1:
        w1, _, left = heappop(heap)
        w2, _, right = heappop(heap)
        heappush(heap, (w1 + w2, next(order), (left, right)))
    
    codes = {}
    def walk(node, prefix=""):
        if isinstance(node, tuple):
            walk(node[0], prefix + "0")
            walk(node[1], prefix + "1")
        else:
            codes[node] = prefix or "0"
    
    walk(heap[0][2])
    return codes


texto = Path("dados-CSM-TP3-Huffman/DecUniversalDH.txt").read_text(encoding="utf-8")
freq = Counter(texto)
total = sum(freq.values())
probs = {symbol: count / total for symbol, count in freq.items()}
codes = gen_huff_table(probs)

print(f"{'simbolo':>10} {'prob':>8} {'codigo'}")
for symbol, prob in sorted(probs.items(), key=lambda x: (-x[1], str(x[0]))):
    print(f"{repr(symbol):>10} {prob:8.4f} {codes[symbol]}")

   simbolo     prob codigo
       ' '   0.1632 110
       'e'   0.0991 001
       'a'   0.0859 000
       'o'   0.0841 1110
       'i'   0.0635 1010
       's'   0.0632 1000
       'd'   0.0536 0110
       'r'   0.0531 0101
       't'   0.0416 11110
       'n'   0.0389 10110
       'm'   0.0296 10010
       'u'   0.0280 01110
       'c'   0.0249 01000
       'l'   0.0222 111111
       'p'   0.0203 101111
      '\n'   0.0139 011110
       'g'   0.0101 1111100
       ','   0.0100 1011101
       'v'   0.0095 1011100
       '.'   0.0078 1001100
       'ç'   0.0078 0111111
       'ã'   0.0075 0111110
       'f'   0.0062 0100101
       'b'   0.0060 0100100
       'q'   0.0049 10011111
       'h'   0.0040 10011011
       'í'   0.0036 01001111
       'A'   0.0035 01001101
       'é'   0.0027 111110111
       'T'   0.0026 111110101
       'º'   0.0026 111110100
       'à'   0.0022 100111001
       'õ'   0.0022 100111010
       'á'   0.0020 100111000
       'z'   0.0019 100110100
       'j'   0.

## Exercício 2


In [ ]:
def encode_huff(message, codes):
    """Codifica uma mensagem usando a tabela de Huffman.
    
    Args:
        message: string com a mensagem a codificar
        codes: dicionario {simbolo: codigo_binario}
    
    Returns:
        string com bits codificados (sequencia de 0s e 1s)
    """
    encoded = ""
    for symbol in message:
        if symbol not in codes:
            raise ValueError(f"Símbolo '{symbol}' não encontrado na tabela de Huffman")
        encoded += codes[symbol]
    return encoded


# Teste da função com a mensagem do arquivo
mensagem_teste = texto[:100]  # Pega os primeiros 100 caracteres
encoded = encode_huff(mensagem_teste, codes)

Mensagem original: '\nDeclaração Universal dos Direitos Humanos\nPreâmbulo\n\nConsiderando que o reconhecimento da dignidade'
Tamanho original: 800 bits (8 bits por símbolo)
Tamanho codificado: 478 bits
Taxa de compressão: 59.75%

Primeiros 100 bits codificados: 0111100100111011001010001111110000101000011111101111101110110100111101111011010101011100001010110000


## Exercício 3

In [ ]:
def decode_huff(encoded_message, codes):
    """Descodifica uma mensagem codificada com Huffman.
    
    Args:
        encoded_message: string com bits codificados (sequencia de 0s e 1s)
        codes: dicionario {simbolo: codigo_binario}
    
    Returns:
        string com a mensagem descodificada (símbolos originais)
    """
    # Inverte a tabela: código -> símbolo
    reverse_codes = {code: symbol for symbol, code in codes.items()}
    
    decoded = ""
    current_code = ""
    
    for bit in encoded_message:
        current_code += bit
        if current_code in reverse_codes:
            decoded += reverse_codes[current_code]
            current_code = ""
    
    if current_code:  # Se sobrarem bits no final
        raise ValueError(f"Código inválido no final da mensagem: '{current_code}'")
    
    return decoded


# Teste: verificar que decode_huff(encode_huff(msg)) == msg
mensagem_original = texto[:100]
encoded = encode_huff(mensagem_original, codes)
decoded = decode_huff(encoded, codes)


Mensagem original: '\nDeclaração Universal dos Direitos Humanos\nPreâmbulo\n\nConsiderando que o reconhecimento da dignidade'
Mensagem descodificada: '\nDeclaração Universal dos Direitos Humanos\nPreâmbulo\n\nConsiderando que o reconhecimento da dignidade'

Mensagens são iguais: True
Comprimento original: 100
Comprimento descodificado: 100


## Exercício 4

In [ ]:
# Exercício 4: Codificar a tabela de Huffman

def encode_table(codes):
    """Codifica a tabela de Huffman numa sequência de bits.
    
    Formato da sequência binária:
    - 16 bits: número de símbolos
    - Para cada símbolo:
        * 8 bits: valor ASCII do símbolo
        * 8 bits: comprimento do código Huffman
        * N bits: o código Huffman em si
    
    Args:
        codes: dicionário {símbolo: código_binário}
    
    Returns:
        string com bits (0s e 1s) da tabela codificada
    """
    encoded_table = ""
    
    # Adiciona número de símbolos (16 bits)
    encoded_table += format(len(codes), '016b')
    
    # Codifica cada símbolo e seu código
    for symbol, code in codes.items():
        encoded_table += format(ord(symbol), '08b')  # Símbolo em 8 bits
        encoded_table += format(len(code), '08b')     # Comprimento do código em 8 bits
        encoded_table += code                        # Código Huffman
    
    return encoded_table


# Codificar a tabela
encoded_table = encode_table(codes)

# Acrescentar a sequência binária da tabela à mensagem codificada (do Ex. 2)
# Resultado final: [tabela codificada] + [mensagem codificada]
full_encoded = encoded_table + encoded


Tabela Huffman codificada: 1586 bits
Primeiros 64 bits: 0000000001000000011000010000001100001100101000000110010110001100


## Exercício 5

In [12]:
def write2file(bits, filename):
    """Escreve uma sequência de bits num ficheiro binário.

    O ficheiro é gravado como bytes e inclui um cabeçalho de 3 bits com o
    número de stuff bits adicionados no fim para completar o último byte.

    Args:
        bits: sequência de bits codificada (string, lista ou tuplo)
        filename: nome do ficheiro de saída

    Returns:
        tuple(Path, int): caminho do ficheiro escrito e número de stuff bits
    """
    if isinstance(bits, (list, tuple)):
        bits = ''.join(str(bit) for bit in bits)
    else:
        bits = str(bits)

    if any(bit not in '01' for bit in bits):
        raise ValueError("A sequência deve conter apenas '0' e '1'.")

    stuff_bits = (8 - ((3 + len(bits)) % 8)) % 8
    header = format(stuff_bits, '03b')
    payload_bits = header + bits + ('0' * stuff_bits)
    payload_bytes = int(payload_bits, 2).to_bytes(len(payload_bits) // 8, byteorder='big')

    output_path = Path(filename)
    output_path.write_bytes(payload_bytes)
    return output_path, stuff_bits

## Exercício 6

In [13]:
def read_file(filename):
    """Lê um ficheiro binário com bits codificados e separa tabela e mensagem.

    O formato esperado é:
    - 3 bits iniciais: número de stuff bits no fim
    - tabela Huffman codificada
    - mensagem codificada
    - stuff bits de preenchimento no fim

    Args:
        filename: nome do ficheiro a ler

    Returns:
        tuple(dict, str): tabela descodificada {simbolo: codigo} e mensagem codificada
    """
    input_path = Path(filename)
    raw_bytes = input_path.read_bytes()
    bitstream = ''.join(format(byte, '08b') for byte in raw_bytes)

    if len(bitstream) < 3:
        raise ValueError("Ficheiro inválido: não contém o cabeçalho de 3 bits.")

    stuff_bits = int(bitstream[:3], 2)
    if stuff_bits > len(bitstream) - 3:
        raise ValueError("Ficheiro inválido: número de stuff bits inconsistente.")

    payload_bits = bitstream[3:len(bitstream) - stuff_bits] if stuff_bits else bitstream[3:]

    if len(payload_bits) < 16:
        raise ValueError("Ficheiro inválido: tabela Huffman incompleta.")

    n_symbols = int(payload_bits[:16], 2)
    index = 16
    decoded_table = {}

    for _ in range(n_symbols):
        if index + 16 > len(payload_bits):
            raise ValueError("Ficheiro inválido: entrada de tabela incompleta.")

        symbol_bits = payload_bits[index:index + 8]
        code_len = int(payload_bits[index + 8:index + 16], 2)
        index += 16

        if index + code_len > len(payload_bits):
            raise ValueError("Ficheiro inválido: código Huffman incompleto.")

        code = payload_bits[index:index + code_len]
        index += code_len
        decoded_table[chr(int(symbol_bits, 2))] = code

    encoded_message = payload_bits[index:]
    return decoded_table, encoded_message

## Exercício 7
### Alínea a

### Alínea b

### Alínea c

### Alínea d

### Alínea e

### Alínea f

### Alínea g

In [ ]:
# Huffman: generate table, encode message, decode message, and encode table
from collections import Counter
import heapq


def gen_huff_table(hist):
    """Generate Huffman codes from a histogram dict {symbol: count}.
    Returns a dict {symbol: code} where code is a string of '0'/'1'."""
    heap = []
    for sym, freq in hist.items():
        heapq.heappush(heap, (freq, sym))
    if len(heap) == 0:
        return {}
    if len(heap) == 1:
        freq, sym = heap[0]
        return {sym: '0'}
    while len(heap) > 1:
        f1, n1 = heapq.heappop(heap)
        f2, n2 = heapq.heappop(heap)
        heapq.heappush(heap, (f1 + f2, (n1, n2)))
    _, root = heap[0]
    codes = {}

    def _traverse(node, prefix):
        if isinstance(node, tuple):
            left, right = node
            _traverse(left, prefix + '0')
            _traverse(right, prefix + '1')
        else:
            codes[node] = prefix

    _traverse(root, '')
    return codes


def encode_huff(message, codes):
    """Encode a message (sequence of symbols) using `codes` dict."""
    return ''.join(codes[s] for s in message)


def decode_huff(bits, codes):
    """Decode a bitstring using `codes` dict. Returns the decoded message."""
    rev = {v: k for k, v in codes.items()}
    out = []
    buf = ''
    for b in bits:
        buf += b
        if buf in rev:
            out.append(rev[buf])
            buf = ''
    if buf:
        raise ValueError('Leftover bits after decoding')
    return ''.join(out)


def _int_to_bits(n, width):
    return format(n, '0{}b'.format(width))


def _bytes_to_bits(b):
    return ''.join(format(x, '08b') for x in b)


def encode_table(codes):
    """Encode the Huffman table into a bitstring. Format (all lengths in 16 bits):
    [num_entries][sym_len][sym_bytes][code_len][code_bits]..."""
    entries = list(codes.items())
    bits = _int_to_bits(len(entries), 16)
    for sym, code in entries:
        bs = sym.encode('utf-8')
        bits += _int_to_bits(len(bs), 16)
        bits += _bytes_to_bits(bs)
        bits += _int_to_bits(len(code), 16)
        bits += code
    return bits


# Demo / usage and required prints
# Prefer an existing histogram `freq` or message `mensagem_teste` if present in the notebook kernel
try:
    hist = freq  # existing Counter in the notebook (if available)
except NameError:
    hist = None

if hist is None:
    try:
        msg = mensagem_teste
        hist = Counter(msg)
    except NameError:
        msg = 'abracadabra'
        hist = Counter(msg)
else:
    # if freq provided, try to recover a message variable, else fall back
    try:
        msg = mensagem_teste
    except NameError:
        # construct a test message from histogram by repeating symbols
        msg = ''.join(sym * count for sym, count in hist.items())

# 1) generate table
codes_new = gen_huff_table(dict(hist))
print('Huffman table (symbol -> code):')
for s, c in codes_new.items():
    print(repr(s), '->', c)

# 2) encode message
encoded_msg = encode_huff(msg, codes_new)
print('\nEncoded message bits:')
print(encoded_msg)

# 3) decode message
decoded_msg = decode_huff(encoded_msg, codes_new)
print('\nDecoded message equals original?:', decoded_msg == msg)
print('Decoded message:')
print(decoded_msg)

# 4) encode table and show combined bits
table_bits = encode_table(codes_new)
full_encoded = table_bits + encoded_msg
print('\nEncoded table bits length:', len(table_bits))
print('Encoded message bits length:', len(encoded_msg))
print('Full encoded (table + message) length:', len(full_encoded))
print('\nEncoded table (bits):')
print(table_bits)
